In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="pRS0coFNudCQWw7NkhTo")
project = rf.workspace("project-dlcnq").project("3d_original")
version = project.version(1)
dataset = version.download("folder")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 3d_original-1 in folder:: 100%|██████████| 1506/1506 [00:00<00:00, 10163.90it/s]


In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.6/914.6 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
# prompt: make /content/Original-Covid-Image-2 folder that only have train folder in that folder split them to train test val for classification model

import os
import shutil
from sklearn.model_selection import train_test_split

# Define paths
original_dataset_dir = '/content/3d_original-1/train'  # Original dataset directory
base_dir = '/content/Original-3D-Covid-Image-Split' # Base directory for the split dataset
os.makedirs(base_dir, exist_ok=True) # creates the directory only if not exists

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# Create directories for train, validation, and test sets
os.makedirs(train_dir, exist_ok=True)
os.makedirs(validation_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Get subdirectories (classes), ignoring hidden directories like '.ipynb_checkpoints'
classes = [d for d in os.listdir(original_dataset_dir) if os.path.isdir(os.path.join(original_dataset_dir, d)) and not d.startswith('.')]

for cls in classes:
  os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
  os.makedirs(os.path.join(validation_dir, cls), exist_ok=True)
  os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

  # Get image filenames for the current class
  class_dir = os.path.join(original_dataset_dir, cls)
  image_files = [f for f in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, f))]

  # Split data, only if there are images in the class directory
  if len(image_files) > 0:  # Check if image_files is not empty
    train_files, test_val_files = train_test_split(image_files, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(test_val_files, test_size=0.5, random_state=42)

    # Copy files to respective directories
    for file in train_files:
        source_path = os.path.join(class_dir, file)
        destination_path = os.path.join(train_dir, cls, file)
        shutil.copy2(source_path, destination_path) # copy2 preserves metadata

    for file in val_files:
        source_path = os.path.join(class_dir, file)
        destination_path = os.path.join(validation_dir, cls, file)
        shutil.copy2(source_path, destination_path)

    for file in test_files:
        source_path = os.path.join(class_dir, file)
        destination_path = os.path.join(test_dir, cls, file)
        shutil.copy2(source_path, destination_path)
  else:
    print(f"Warning: No image files found in directory: {class_dir}") # Print warning for empty directories

In [ ]:
import torch
import random
import numpy as np

def set_random_seed(seed=42):
    """
    กำหนดค่า random seed ให้คงที่
    Args:
        seed: ค่า seed ที่ใช้กำหนด (ค่าเริ่มต้นคือ 42)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch import nn, optim
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO

# Define data transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
train_dir = Path('/content/Original-3D-Covid-Image-Split/train')
test_dir = Path('/content/Original-3D-Covid-Image-Split/validation')

# Delete `.ipynb_checkpoints` from both train and test directories
for folder in [train_dir, test_dir]:
    checkpoints_dir = folder / '.ipynb_checkpoints'
    if checkpoints_dir.exists():
        shutil.rmtree(checkpoints_dir)

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load EfficientNet, ViT, or ConvNeXt models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Choose model (EfficientNet, ViT, or ConvNeXt)
model_name = 'efficientnet'  # Change this to 'vit' or 'convnext' for other models

if model_name == 'efficientnet':
    model = models.efficientnet_b4(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 3)
elif model_name == 'vit':
    from transformers import ViTForImageClassification
    model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k')
    model.classifier = nn.Linear(model.classifier.in_features, 3)
elif model_name == 'convnext':
    model = models.convnext_base(pretrained=True)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, 3)
elif model_name == 'resnet':
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 3)
elif model_name == 'yolo':
    model = YOLO("yolo11n-cls.yaml").load("yolo11n-cls.pt")  # ใช้โมเดลสำหรับ classification
else:
    raise ValueError("Model name must be 'efficientnet', 'vit', 'convnext', 'resnet', or 'yolo'")

model = model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

# Training loop
def train(model, train_loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs.logits, labels) if model_name == 'vit' else criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluation loop
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.logits, 1) if model_name == 'vit' else torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")

# Train the model
set_random_seed(42)

if model_name != 'yolo':
    model = model.to(device)

# YOLO Training Configuration
if model_name == 'yolo':
    results = model.train(data="/content/COVID-19-5", epochs=10, imgsz=224)  # ใช้ imgsz=224 เพื่อให้ขนาดภาพตรงกับโมเดลอื่น ๆ
    model.save("yolo_model.pt")
    model.export(format="torchscript")
else:
    train(model, train_loader, criterion, optimizer, epochs=10)
    torch.save(model.state_dict(), f"{model_name}_model_3d_aug.pth")

# Evaluate the Model
if model_name != 'yolo':
    evaluate(model, test_loader)
else:
    metrics = model.val(data="/content/data.yaml", imgsz=224)
    print(metrics)

Epoch 1/10: 100%|██████████| 33/33 [00:16<00:00,  2.01batch/s, loss=0.0781]


Epoch 1, Loss: 0.5375


Epoch 2/10: 100%|██████████| 33/33 [00:14<00:00,  2.22batch/s, loss=0.117]


Epoch 2, Loss: 0.1649


Epoch 3/10: 100%|██████████| 33/33 [00:15<00:00,  2.19batch/s, loss=0.0208]


...

Epoch 8, Loss: 0.0352


Epoch 9/10: 100%|██████████| 33/33 [00:15<00:00,  2.18batch/s, loss=0.0428]


Epoch 9, Loss: 0.0483


Epoch 10/10: 100%|██████████| 33/33 [00:15<00:00,  2.18batch/s, loss=0.00412]


Epoch 10, Loss: 0.0256
Test Accuracy: 94.67%


In [45]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm
from pathlib import Path
import shutil

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalization
])

test_dir = Path('/content/imege_2000_1/3d_original')

# Delete `.ipynb_checkpoints` from both train and test directories
for folder in [test_dir]:
    checkpoints_dir = folder / '.ipynb_checkpoints'
    if checkpoints_dir.exists():
        shutil.rmtree(checkpoints_dir)

test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = 'efficientnet'
if model_name == 'efficientnet':
    model = models.efficientnet_b4(pretrained=True)
    model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 3)
elif model_name == 'vit':
    from transformers import ViTForImageClassification
    model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k')
    model.classifier = torch.nn.Linear(model.classifier.in_features, 3)
elif model_name == 'convnext':
    model = models.convnext_base(pretrained=True)
    model.classifier[2] = torch.nn.Linear(model.classifier[2].in_features, 3)
elif model_name == 'resnet':
    model = models.resnet50(pretrained=True)
    model.fc = torch.nn.Linear(model.fc.in_features, 3)
elif model_name == 'yolo':
    from ultralytics import YOLO
    model = YOLO("/content/yolo_model.pt")
else:
    raise ValueError("Model name must be 'efficientnet', 'vit', 'convnext', 'resnet', or 'yolo'")

if model_name != 'yolo':
    model.load_state_dict(torch.load(f"{model_name}_model_3d_ori.pth"))
    model = model.to(device)

class_names = ['normal', 'covid', 'pneumonia']

def test_model(model, test_loader):
    if model_name == 'yolo':
        results = model.val(data="/content/data.yaml", imgsz=224)
        accuracy = results.top1
        print(f"YOLO Test Top-1 Accuracy: {accuracy:.2f}%")
        return

    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            logits = outputs.logits if model_name == 'vit' else outputs
            _, predicted = torch.max(logits, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            for i in range(len(images)):
                print(f"True label: {class_names[labels[i]]}, Predicted label: {class_names[predicted[i]]}")

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")


test_model(model, test_loader)

Evaluating:   1%|          | 1/188 [00:01<06:11,  1.99s/it]

True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted la

Evaluating:   1%|          | 2/188 [00:02<04:19,  1.39s/it]

True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted label: normal
True label: normal, Predicted la

Evaluating:   2%|▏         | 3/188 [00:04<04:12,  1.37s/it]

...

True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: covid
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True

Evaluating:  99%|█████████▉| 187/188 [04:16<00:01,  1.17s/it]

True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: normal
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: normal
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True l

Evaluating: 100%|██████████| 188/188 [04:17<00:00,  1.37s/it]

True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
True label: pneumonia, Predicted label: pneumonia
Test Accuracy: 88.63%


In [40]:
import shutil

# Copy files to a writable location
shutil.copytree("/content/drive/MyDrive/imege_2000", "/content/imege_2000", dirs_exist_ok=True)

# Now work with /content/imege_2000 instead of Google Drive
base_dir = "/content/imege_2000"


In [44]:
import os
import shutil

# Define the base directory
base_dir = "/content/imege_2000_1"  # Make sure this matches your actual path

# Define target directories
original_dir = os.path.join(base_dir, "3d_original")
preprocessed_dir = os.path.join(base_dir, "3d_preprocessed")

# Create directories if they don't exist
os.makedirs(original_dir, exist_ok=True)
os.makedirs(preprocessed_dir, exist_ok=True)

# Iterate through the folders in base directory
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)

    # Ensure it's a directory before moving
    if os.path.isdir(folder_path):
        # Move "3d_original_*" folders
        if folder.startswith("3d_original_"):
            shutil.move(folder_path, os.path.join(original_dir, folder))

        # Move "3d_preprocessed_*" folders
        elif folder.startswith("3d_preprocessed_"):
            shutil.move(folder_path, os.path.join(preprocessed_dir, folder))

print("Folders have been successfully separated!")


Folders have been successfully separated!
